<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 19. CNN 이미지 분류: Dataset부터 confusion matrix까지

외부 다운로드 없이 생성된 28×28 도형 이미지를 분류합니다. 직접 작성한 Dataset,
train 전용 augmentation, 소형 CNN, 학습/평가 루프, confusion matrix, checkpoint를
하나의 재현 가능한 파이프라인으로 연결합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 학습 목표

- `(N,H,W)` uint8을 `(N,C,H,W)` float32로 변환한다.
- split 이후 train에만 무작위 augmentation을 적용한다.
- convolution/pooling 뒤 tensor shape를 추론한다.
- accuracy뿐 아니라 class별 confusion matrix를 해석한다.
- 모델 설정과 class 이름을 checkpoint에 함께 저장한다.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
SCALER = ACCELERATOR.grad_scaler()
print(ACCELERATOR.summary())

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 데이터와 stratified split

원본/증강본이 양쪽 분할에 섞이지 않도록 **원본 index를 먼저 분할**합니다.
클래스마다 70/15/15 비율을 사용합니다.

In [ ]:
raw = np.load(DATA / "image_shapes.npz")
images, labels, class_names = raw["images"], raw["labels"], raw["class_names"].tolist()
print(images.shape, images.dtype, labels.shape, class_names)
print("class counts:", np.bincount(labels).tolist())

In [ ]:
def stratified_indices(labels, seed=42):
    rng = np.random.default_rng(seed)
    train, valid, test = [], [], []
    for label in np.unique(labels):
        index = np.flatnonzero(labels == label)
        rng.shuffle(index)
        a, b = int(0.70 * len(index)), int(0.85 * len(index))
        train.extend(index[:a])
        valid.extend(index[a:b])
        test.extend(index[b:])
    return tuple(np.asarray(part, dtype=np.int64) for part in (train, valid, test))


train_idx, valid_idx, test_idx = stratified_indices(labels)
print(len(train_idx), len(valid_idx), len(test_idx))

In [ ]:
assert not (
    set(train_idx) & set(valid_idx)
    | set(train_idx) & set(test_idx)
    | set(valid_idx) & set(test_idx)
)
assert len(train_idx) + len(valid_idx) + len(test_idx) == len(labels)
for part in (train_idx, valid_idx, test_idx):
    assert len(np.unique(labels[part])) == len(class_names)
print("split contract: OK")

## 2) custom Dataset과 augmentation

수평/수직 flip, 약한 Gaussian noise를 직접 구현합니다. 이 합성 도형에서는 방향이
class 의미를 바꾸지 않지만, 숫자 6/9처럼 방향이 label인 문제에는 이런 증강이 위험합니다.

In [ ]:
class ShapesDataset(Dataset):
    def __init__(self, images, labels, indices, augment=False):
        self.images, self.labels = images, labels
        self.indices = np.asarray(indices)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = int(self.indices[item])
        image = (
            torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0) / 255.0
        )
        if self.augment:
            if torch.rand(()) < 0.5:
                image = torch.flip(image, [2])
            if torch.rand(()) < 0.5:
                image = torch.flip(image, [1])
            image = (image + torch.randn_like(image) * 0.025).clamp(0, 1)
        return image, torch.tensor(self.labels[index], dtype=torch.long)

In [ ]:
train_loader = DataLoader(
    ShapesDataset(images, labels, train_idx, True),
    batch_size=64,
    shuffle=True,
    pin_memory=ACCELERATOR.pin_memory,
)
valid_loader = DataLoader(
    ShapesDataset(images, labels, valid_idx),
    batch_size=128,
    pin_memory=ACCELERATOR.pin_memory,
)
test_loader = DataLoader(
    ShapesDataset(images, labels, test_idx),
    batch_size=128,
    pin_memory=ACCELERATOR.pin_memory,
)
bx, by = next(iter(train_loader))
print(bx.shape, bx.dtype, by.shape, bx.min().item(), bx.max().item())
assert bx.shape[1:] == (1, 28, 28) and bx.dtype == torch.float32

## 3) 작은 CNN

3×3 convolution은 지역 패턴을 찾고, pooling은 해상도를 줄이며 수용 영역을 넓힙니다.
`AdaptiveAvgPool2d(1)`을 사용하면 flatten 차원을 손으로 계산하는 오류가 줄어듭니다.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 24, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(24, num_classes)

    def forward(self, x):
        features = self.features(x).flatten(1)
        return self.classifier(features)


model = ACCELERATOR.move(TinyCNN(len(class_names)))
probe = model(ACCELERATOR.move(bx[:4]))
print(model, "\nlogits:", probe.shape)

In [ ]:
assert probe.shape == (4, len(class_names))
parameter_count = sum(p.numel() for p in model.parameters())
print("parameters:", parameter_count)
assert parameter_count < 20_000

## 4) 학습과 validation checkpoint 선택

validation accuracy가 개선될 때만 state를 메모리에 보관합니다. test는 모델 선택에
사용하지 않고 마지막에 한 번 평가합니다.

In [ ]:
def train_cnn_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = ACCELERATOR.move(x, y)
        optimizer.zero_grad(set_to_none=True)
        with ACCELERATOR.autocast():
            loss = nn.functional.cross_entropy(model(x), y)
        SCALER.scale(loss).backward()
        SCALER.step(optimizer)
        SCALER.update()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.inference_mode()
def predict_cnn(model, loader):
    model.eval()
    predictions, targets = [], []
    for x, y in loader:
        predictions.append(model(ACCELERATOR.move(x)).argmax(1).cpu())
        targets.append(y)
    return torch.cat(predictions).numpy(), torch.cat(targets).numpy()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=4e-3)
best_accuracy, best_state = -1.0, None
for epoch in range(4):
    loss = train_cnn_epoch(model, train_loader, optimizer)
    valid_pred, valid_true = predict_cnn(model, valid_loader)
    accuracy = float((valid_pred == valid_true).mean())
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_state = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
    print(f"epoch={epoch + 1} loss={loss:.4f} valid_accuracy={accuracy:.3f}")
model.load_state_dict(best_state)

## 5) confusion matrix

행은 실제 class, 열은 예측 class입니다. 전체 accuracy가 같아도 특정 class 쌍이
반복적으로 혼동되는지 확인하면 데이터/모델 개선 방향을 찾을 수 있습니다.

In [ ]:
test_pred, test_true = predict_cnn(model, test_loader)
matrix = confusion_matrix(test_true, test_pred, labels=np.arange(len(class_names)))
matrix_frame = pd.DataFrame(
    matrix,
    index=[f"true_{x}" for x in class_names],
    columns=[f"pred_{x}" for x in class_names],
)
test_accuracy = float((test_pred == test_true).mean())
print(matrix_frame)
print("test accuracy:", round(test_accuracy, 3))
assert matrix.sum() == len(test_idx) and np.isfinite(test_accuracy)

In [ ]:
per_class_recall = matrix.diagonal() / matrix.sum(axis=1).clip(min=1)
for name, recall in zip(class_names, per_class_recall):
    print(f"{name:>8s}: recall={recall:.3f}")
weakest = class_names[int(per_class_recall.argmin())]
print("가장 어려운 class:", weakest)

## 6) checkpoint와 단일 이미지 추론

배포 시 class index 순서를 잃으면 모델이 맞는 index를 내도 잘못된 이름으로 표시됩니다.
따라서 `class_names`와 입력 정규화 정보를 모델과 함께 저장합니다.

In [ ]:
checkpoint_path = ARTIFACTS / "tiny_shapes_cnn_solution.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "class_names": class_names,
        "input_shape": [1, 28, 28],
        "scale": 255.0,
    },
    checkpoint_path,
)
payload = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
restored = ACCELERATOR.move(TinyCNN(len(payload["class_names"])))
restored.load_state_dict(payload["state_dict"])
restored.eval()
sample_image, sample_label = test_loader.dataset[0]
with torch.inference_mode():
    sample_index = (
        restored(ACCELERATOR.move(sample_image.unsqueeze(0))).argmax(1).item()
    )
print(
    "actual:", class_names[sample_label.item()], "predicted:", class_names[sample_index]
)

In [ ]:
assert checkpoint_path.exists()
assert payload["class_names"] == class_names
with torch.inference_mode():
    sample_batch = ACCELERATOR.move(sample_image.unsqueeze(0))
    before = model(sample_batch)
    after = restored(sample_batch)
assert torch.allclose(before, after)
print("checkpoint contract: OK")

## 오류 분석 과제

test에서 틀린 index 5개를 찾아 원본 픽셀 통계(평균/최댓값)와 실제·예측 class를
출력하세요. 그 뒤 noise 크기, 채널 수, epoch 중 하나만 바꾸어 confusion matrix가
어떻게 달라지는지 비교하세요.

In [ ]:
mistakes = np.flatnonzero(test_pred != test_true)[:5]
for position in mistakes:
    original_index = test_idx[position]
    print(
        {
            "dataset_index": int(original_index),
            "pixel_mean": round(float(images[original_index].mean()), 2),
            "actual": class_names[int(test_true[position])],
            "predicted": class_names[int(test_pred[position])],
        }
    )
print("mistake count:", int((test_pred != test_true).sum()))

## 마무리

이미지 파이프라인의 재현성은 모델만이 아니라 split index, train-only augmentation,
dtype/range, class 순서까지 포함합니다. 실제 대형 이미지에서는 파일을 지연 로딩하고
DataLoader worker/pinned memory를 측정해 조정하세요.